# EcoShot AI - E6 detector training (Google Colab, T4 GPU)
Runtime > Change runtime type > **T4 GPU** before you start. Run cells top to bottom.

Rules: choose settings on **val** only. Run the **test** split once at the very end. Nothing here fabricates results: every number comes from the files these cells write.

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 1. Get the code
**Option A (after the branch is pushed):** clone `feature/e6-detector`. **Option B:** unzip `ecoshot_e6_detector_code.zip` from your Drive. Use one of them.

In [ ]:
import os
os.chdir('/content')
# Option A
# !git clone --branch feature/e6-detector https://github.com/prateekchaurasia11-lab/EcoSort-AI.git repo
# Option B (edit the path to where you saved the zip)
# !unzip -q -o /content/drive/MyDrive/EcoSort-AI-Backups/ecoshot_e6_detector_code.zip -d repo
%cd /content/repo
!ls Src/detector

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Copy the dataset to the fast local disk
Put the frozen YOLO dataset (images/, labels/, data.yaml) in a zip on Drive. **data.yaml must use an absolute `path:`**. After unzipping here, set it to `/content/det_data`.

In [ ]:
!mkdir -p /content/det_data
!unzip -q -o "/content/drive/MyDrive/EcoSort-AI-Backups/det_dataset_v1.zip" -d /content/det_data
import yaml, pathlib
p = pathlib.Path('/content/det_data/data.yaml'); cfg = yaml.safe_load(p.read_text())
cfg['path'] = '/content/det_data'; p.write_text(yaml.safe_dump(cfg)); print(cfg)

## 3. Check the dataset (must say PASS)
Add `--manifest groups.csv` (columns `file_name,group_id`) once Prince provides real group IDs.

In [ ]:
!python -m Src.detector.check_dataset --data /content/det_data/data.yaml --report /content/det_check.json

## 4. Train (direct six-class YOLOv8n)
First run: keep these settings so results are comparable. Change one thing at a time and record why.

In [ ]:
!python -m Src.detector.train_yolo --data /content/det_data/data.yaml --model yolov8n.pt \
    --epochs 100 --imgsz 640 --batch 16 --seed 42 --project /content/runs/e6 --name e6_yolov8n_direct6

## 5. Choose the detection threshold on VAL
Try a few values, compare precision/recall, pick one. Then write the protocol with that value.

In [ ]:
RUN = '/content/runs/e6/e6_yolov8n_direct6'
!python -m Src.detector.make_protocol --checkpoint {RUN}/weights/best.pt --out /content/protocol.json --conf 0.25 --run-manifest {RUN}/run_manifest.json
for c in (0.15, 0.25, 0.35):
    print('=== conf', c)
    !python -m Src.detector.evaluate_detector --checkpoint {RUN}/weights/best.pt --protocol /content/protocol.json --data /content/det_data/data.yaml --split val --conf {c} --skip-ultralytics-val --out-dir /content/results/val_conf_{c}

In [ ]:
# After choosing the conf, rewrite protocol.json with it (example: 0.25), then run the full validation report:
CONF = 0.25
!python -m Src.detector.make_protocol --checkpoint {RUN}/weights/best.pt --out /content/protocol.json --conf {CONF} --run-manifest {RUN}/run_manifest.json
!python -m Src.detector.evaluate_detector --checkpoint {RUN}/weights/best.pt --protocol /content/protocol.json --data /content/det_data/data.yaml --split val --out-dir /content/results/e6_val

## 6. Save everything to Drive (do this before the runtime disconnects)

In [ ]:
!mkdir -p /content/drive/MyDrive/EcoSort-AI-Backups/e6_run1
!cp -r {RUN} /content/drive/MyDrive/EcoSort-AI-Backups/e6_run1/
!cp /content/protocol.json /content/det_check.json /content/drive/MyDrive/EcoSort-AI-Backups/e6_run1/
!cp -r /content/results /content/drive/MyDrive/EcoSort-AI-Backups/e6_run1/
!sha256sum {RUN}/weights/best.pt

## 7. TEST split - run ONCE, only after the threshold and checkpoint are final
Do not change anything after seeing these numbers. If you do, the test split is no longer independent.

In [ ]:
# !python -m Src.detector.evaluate_detector --checkpoint {RUN}/weights/best.pt --protocol /content/protocol.json --data /content/det_data/data.yaml --split test --out-dir /content/results/e6_test
# !cp -r /content/results/e6_test /content/drive/MyDrive/EcoSort-AI-Backups/e6_run1/